# DeepDetect AI ? complete end-to-end project notebook

This notebook is the executable project record, from imports and dataset auditing through SCRFD face preprocessing, model training, held-out evaluation, performance comparison, explainability, FastAPI integration, and the React frontend.

**Labels:** `0 = real`, `1 = fake`  
**Production detector:** SCRFD through InsightFace  
**Candidates:** ResNet-18 + BiLSTM baseline, EfficientNet-V2-S + BiLSTM, MobileNet-V3 + XGBoost  
**Production artifact:** MobileNet-V3 + XGBoost with pretrained ImageNet features  

The application is API-first. The removed Streamlit frontend is not part of this notebook.


## 0. How to use this notebook

Run the cells from top to bottom. The default configuration reads the existing trained artifacts and benchmark report, so the notebook is quick and reproducible. Set `RUN_FULL_TRAINING = True` only when you intentionally want to retrain all candidates.

The full CPU-safe benchmark used 150 real and 150 fake videos, eight sampled frames, seed 42, and an identical stratified split. Training can take a long time on CPU; a GPU is recommended for new experiments.


In [1]:
# Install into the active Jupyter kernel. Run this cell once, then restart
# the kernel so newly installed native packages are importable.
%pip install -r requirements.txt

import base64
import json
import os
import sys
from pathlib import Path

import importlib.util
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, roc_curve, auc

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts_full"
CACHE_DIR = PROJECT_ROOT / "data_cache"
SEED = 42
NUM_FRAMES = 8
LIMIT_PER_CLASS = 150
RUN_FULL_TRAINING = False
RUN_SAMPLE_INFERENCE = False

np.random.seed(SEED)
torch.manual_seed(SEED)
missing = [name for name in ("insightface", "onnxruntime", "cv2") if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(
        "Missing packages in the active notebook kernel: " + ", ".join(missing) + ". "
        "Run the install command above, restart the kernel, and rerun this cell."
    )
print("Project:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


Project: c:\Users\Nishant\OneDrive\Documents\deep_detect
Python: 3.12.13
PyTorch: 2.13.0+cpu
Device: cpu


## 1. Project and dependency audit

The production dependencies include PyTorch/torchvision, InsightFace, ONNX Runtime, OpenCV, scikit-learn, XGBoost, FastAPI, ReportLab, and the React/Vite frontend toolchain.


In [2]:
from src.face_detector import get_detector_name
from src.models_zoo import SUPPORTED_MODEL_NAMES

required_paths = [
    PROJECT_ROOT / "src" / "api.py",
    PROJECT_ROOT / "src" / "preprocessing.py",
    PROJECT_ROOT / "src" / "face_detector.py",
    PROJECT_ROOT / "src" / "models_zoo.py",
    PROJECT_ROOT / "src" / "benchmark.py",
    PROJECT_ROOT / "frontend" / "src" / "App.tsx",
]
print("Detector:", get_detector_name())
print("Supported models:", SUPPORTED_MODEL_NAMES)
for path in required_paths:
    print(f"{path.relative_to(PROJECT_ROOT)}: {'OK' if path.exists() else 'MISSING'}")


RuntimeError: SCRFD is unavailable. Install insightface and onnxruntime, or set DEEPDETECT_ALLOW_HAAR=1 only for local development.

## 2. Dataset audit and label construction

`collect_items` searches the project dataset folders and creates balanced labels:

- `Celeb-real`, `YouTube-real` ? label `0`
- `Celeb-synthesis` ? label `1`

The same list and seed are used before splitting so every candidate is evaluated on the same videos.


In [ ]:
from src.benchmark import collect_items

items = collect_items(str(DATA_DIR), limit_per_class=LIMIT_PER_CLASS, seed=SEED)
label_table = pd.Series([label for _, label in items]).map({0: "real", 1: "fake"}).value_counts().rename_axis("label").reset_index(name="videos")
print("Total videos:", len(items))
display(label_table)
print("Example:", items[0])


## 3. SCRFD face detection and temporal preprocessing

Each video is uniformly sampled into a fixed-length sequence. SCRFD detects the primary face, applies the project crop margin, resizes crops to `224 ? 224`, and stores reusable results in `data_cache`. Haar Cascade is not used unless explicitly enabled for development.


In [ ]:
from src.preprocessing import process_video

sample_path = items[0][0]
face_sequence, face_images = process_video(sample_path, num_frames=NUM_FRAMES)
print("Video:", sample_path)
print("Face tensor/array shape:", face_sequence.shape)
print("Number of face images:", len(face_images))

fig, axes = plt.subplots(1, min(len(face_images), 8), figsize=(16, 3))
for index, (axis, image) in enumerate(zip(np.atleast_1d(axes), face_images[:8])):
    axis.imshow(image)
    axis.set_title(f"Frame {index + 1}")
    axis.axis("off")
plt.tight_layout()


## 4. Model architectures

### Baseline
**ResNet-18 + BiLSTM** remains the reference model for the project.

### Candidate 2
**EfficientNet-V2-S + BiLSTM** tests a stronger modern spatial backbone while retaining temporal modeling.

### Candidate 3
**MobileNet-V3 + XGBoost** extracts pretrained frame features, aggregates mean and standard deviation across time, and classifies the video with XGBoost. It is lightweight, but it does not expose gradients for native Grad-CAM.


In [ ]:
from src.models_zoo import build_sequence_model

for model_name in SUPPORTED_MODEL_NAMES:
    model = build_sequence_model(model_name, pretrained=False)
    print(f"{model_name:28} {type(model).__name__}")


## 5. Reproducible split and training configuration

The benchmark uses a stratified train/test split, followed by a stratified validation split from the training portion. The test set remains untouched until final evaluation. All candidates use the same split.

The training cell is guarded by `RUN_FULL_TRAINING` to avoid accidentally starting a long CPU job.


In [ ]:
from sklearn.model_selection import train_test_split

train_items, test_items = train_test_split(
    items, test_size=0.25, random_state=SEED,
    stratify=[label for _, label in items],
)
train_items, validation_items = train_test_split(
    train_items, test_size=0.20, random_state=SEED,
    stratify=[label for _, label in train_items],
)
print({
    "train": len(train_items),
    "validation": len(validation_items),
    "test": len(test_items),
    "seed": SEED,
    "frames_per_video": NUM_FRAMES,
})


In [ ]:
from src.benchmark import run_benchmark

if RUN_FULL_TRAINING:
    training_output = run_benchmark(
        data_dir=str(DATA_DIR),
        output_dir=str(ARTIFACT_DIR),
        limit_per_class=LIMIT_PER_CLASS,
        num_frames=NUM_FRAMES,
        epochs=1,
        batch_size=2,
        pretrained=True,
        seed=SEED,
    )
    print(json.dumps(training_output, indent=2))
else:
    print("Training skipped. Set RUN_FULL_TRAINING = True to retrain all candidates.")


## 6. Performance matrix and model selection

The primary selection metric is held-out **balanced accuracy**, with ROC-AUC as the tie-breaker. Accuracy, F1, and ROC-AUC are also reported. Balanced accuracy is important because a model can achieve misleading accuracy by favoring one class.


In [ ]:
report_path = ARTIFACT_DIR / "benchmark_results.json"
if not report_path.exists():
    raise FileNotFoundError(f"Missing benchmark report: {report_path}")
report = json.loads(report_path.read_text(encoding="utf-8"))
metrics = pd.DataFrame(report["results"]).T
metrics = metrics[["accuracy", "balanced_accuracy", "f1", "roc_auc"]].sort_values(
    "balanced_accuracy", ascending=False
)
print("Benchmark winner:", report.get("winner"))
display((metrics * 100).round(2).style.format("{:.2f}%"))

ax = (metrics * 100).plot.bar(figsize=(11, 5), ylim=(0, 100), rot=20)
ax.set_ylabel("Score (%)")
ax.set_title("Held-out performance comparison")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()


In [ ]:
artifact_paths = {
    "resnet18_bilstm": ARTIFACT_DIR / "resnet18_bilstm.pt",
    "efficientnet_v2_bilstm": ARTIFACT_DIR / "efficientnet_v2_bilstm.pt",
    "mobilenet_xgboost": ARTIFACT_DIR / "mobilenet_xgboost.joblib",
}
selected_model = report["winner"]
selected_artifact = artifact_paths[selected_model]
print("Selected model:", selected_model)
print("Selected artifact:", selected_artifact)
print("Artifact exists:", selected_artifact.exists())
print("Production training note:", report.get("production_training", "not recorded"))


## 7. Metric definitions and confusion-matrix workflow

The report stores aggregate metrics. For a new experiment, keep the held-out labels and probabilities and run the following cell to produce a classification report, confusion matrix, ROC curve, and AUC. This is the required evaluation workflow for future training runs.


In [ ]:
# Replace these two arrays with predictions from a new held-out run.
# y_test = np.array([...], dtype=int)
# y_probability = np.array([...], dtype=float)
# y_pred = (y_probability >= 0.5).astype(int)
# print(classification_report(y_test, y_pred, target_names=["real", "fake"], digits=4))
# ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["real", "fake"], cmap="Blues")
# plt.show()
# fpr, tpr, _ = roc_curve(y_test, y_probability)
# print("ROC-AUC:", auc(fpr, tpr))
# plt.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.3f}")
# plt.plot([0, 1], [0, 1], "--", color="gray")
# plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.legend(); plt.show()
print("Evaluation template ready: provide y_test and y_probability from a held-out split.")


## 8. Production inference and explainability

The API loads the selected artifact through `src.inference`. For every completed job it returns the video score, real/fake probabilities, sampled face frames, timestamps, and suspicious-frame evidence.

- ResNet/EfficientNet candidates can use native Grad-CAM when their PyTorch model is active.
- MobileNet-XGBoost has no gradients, so the API uses a fast visual artifact heatmap and labels it as an evidence cue rather than native Grad-CAM.
- Suspicious frames are tied to their sample index, timestamp, score, face image, and explanation text.


In [ ]:
from src.inference import load_detector_model

loaded_model, device = load_detector_model(
    model_path=str(selected_artifact),
    backbone_type=selected_model,
)
print("Loaded:", type(loaded_model).__name__)
print("Device:", device)

if RUN_SAMPLE_INFERENCE:
    from src.inference import predict_video
    inference_result = predict_video(
        video_path=sample_path,
        model=loaded_model,
        device=device,
        num_frames=NUM_FRAMES,
        backbone_type=selected_model,
    )
    print({key: inference_result[key] for key in ("verdict", "fake_probability", "real_probability", "num_frames")})
else:
    print("Sample inference skipped. Set RUN_SAMPLE_INFERENCE = True to run it.")


## 9. FastAPI backend verification

FastAPI is the current application backend and serves the built React frontend from `frontend/dist`. It provides authentication, upload/URL/batch analysis, polling, video streaming, compatible preview fallback, suspicious evidence, and PDF reports.


In [ ]:
from fastapi.testclient import TestClient
from src.api import app

client = TestClient(app)
for endpoint in ("/api/health", "/api/models"):
    response = client.get(endpoint)
    print(endpoint, response.status_code, response.json())


## 10. Run the final application

From the repository root:

```powershell
cd frontend
npm run build
cd ..
python -m uvicorn src.api:app --host 0.0.0.0 --port 8000
```

Open `http://127.0.0.1:8000`.

Important API routes:

- `GET /api/health`
- `GET /api/models`
- `POST /api/analyses/upload?num_frames=32`
- `POST /api/analyses/url`
- `GET /api/analyses/{job_id}`
- `GET /api/analyses/{job_id}/video`
- `GET /api/analyses/{job_id}/preview`
- `GET /api/analyses/{job_id}/report`


## 11. Reproducibility, limitations, and interpretation

- Seed: `42`.
- Labels are balanced only within the configured sample; they do not represent every available dataset video.
- The benchmark is a project comparison, not a publication-quality forensic validation.
- The deployed pretrained-feature MobileNet check was approximately 60.00% accuracy, 60.14% balanced accuracy, F1 0.634, and ROC-AUC 0.659.
- A model score is a screening signal, not proof that a video is authentic or manipulated.
- More data, longer GPU training, calibration, codec diversity, and an external test set are needed for stronger claims.
- SCRFD assets are downloaded by InsightFace on first use. Haar fallback is disabled unless `DEEPDETECT_ALLOW_HAAR=1` is explicitly set for development.
